# Q-Learning CIA Test 1 — FrozenLake Debugging
**Marks:15**
## Instructions

The following program implements a **Q-learning delivery robot using a custom 4 × 4 FrozenLake environment**.

- Start state = 3
- Goal state = 12
- Correct the errors without completely rewriting the program.
- Execute the corrected program and verify the learned route and final reward.

### Environment

```text
F F F S
F F H F
F F F F
G F F F
```
State numbering is row-wise from 0 to 15.

In [4]:
# Q-Learning using a custom FrozenLake environment

import numpy as np
import random
import gymnasium as gym

custom_map = [
    "FFFS",
    "FFHF",
    "FFFF",
    "GFFF"
]

env = gym.make(
    "FrozenLake-v1",
    desc=custom_map,
    is_slippery=True
)

n_states = env.observation_space.n
n_actions = env.action_space.n

Q_table = np.ones((n_states, n_actions))

learning_rate = 0.01
discount_factor = 0.50
epsilon = 0.2
episodes = 100

# 0 = Left, 1 = Down, 2 = Right, 3 = Up

def choose_action(state):
    # BUG :So basically here if random.random() > epsilon with np.argmin for the best action then it should explore always instead of  rarely 
    #because of which it will ultimately choose worst option and we dont want it so , fix is 
    # we just need to flip it to lesser then symbol and use argmax - best, not argmin worst
    if random.random() < epsilon:
        return random.randrange(n_actions)
    else:
        return np.argmax(Q_table[state])

for episode in range(episodes):

    state, info = env.reset()

    for step in range(100):

        action = choose_action(state)

        next_state, reward, terminated, truncated, info = env.step(action)

        # BUG:terminated and truncated. So basically it ends if either happens fix is 
        # we have to chnage and to or 
        done = terminated or truncated

        # BUG:np.min basically same as above it should have best value not worst
        # FIX:we will again change min to max to get best results np.min -> np.max.
        best_next_q = np.max(Q_table[next_state])

        # BUG: multiplying by learning_rate and basically in Q-learning formula it multiplies future value by discount_factor not learning_rate
        # FIX: we will change learning rate to discount factor 
        target = reward + discount_factor * best_next_q

        if done:
            target = reward

        # BUG: Here Q_table[next_state, action] , So We must update the q value to state not the next state 
        # FIX: we will change next_state to state
        Q_table[state, action] = (
            Q_table[state, action]
            + learning_rate *
            (target - Q_table[state, action])
        )

        state = next_state

        if done:
            break

print("Learned Q-table:")
print(np.round(Q_table, 3))

action_symbols = {
    0: "←",
    1: "↓",
    2: "→",
    3: "↑"
}

grid = env.unwrapped.desc

print("\nLearned Delivery Robot Policy")
print("-----------------------------")

for row in range(4):

    line = ""

    for col in range(4):

        state = row * 4 + col
        cell = grid[row][col].decode()

        if cell == "S":
            symbol = "S"
        elif cell == "G":
            symbol = "G"
        elif cell == "H":
            symbol = "X"
        else:
            # BUG: np.argmin Same as above we will change it to argmax.
            # FIX: chnge argmin to argmax to get the best results
            best_action = np.argmax(Q_table[state])
            symbol = action_symbols[best_action]

        line += f"[ {symbol} ] "

    print(line)

state, info = env.reset()

start_state = state
path = [state]
total_reward = 0

for step in range(30):

    # BUG was: np.argmin same as above , we will change it to argmax 
    # FIX: So we will chnage the argmin to argmax to get best results instead of getting worst
    action = np.argmax(Q_table[state])

    next_state, reward, terminated, truncated, info = env.step(action)

    total_reward += reward
    path.append(next_state)

    state = next_state

    # BUG :terminated and truncated which was already fixed in above statements , so here also we will do the same.
    # FIX: we will change and to or 
    if terminated or truncated:
        break

print("\nStart state:", start_state)
print("Path followed:", path)
print("Total reward:", total_reward)

# BUG: so here bug is that goal_state = 15. which is incorrect because according to the map it should be at 12
# FIX: according to map G is at 12 not 15 so instead of goal_state =15 we will change it to 12 
goal_state = 12

if state == goal_state:
    print("Delivery successful!")
else:
    print("Delivery unsuccessful.")

env.close()


Learned Q-table:
[[0.914 0.91  0.91  0.914]
 [0.905 0.902 0.908 0.907]
 [0.796 0.795 0.792 0.797]
 [0.665 0.666 0.664 0.663]
 [0.965 0.961 0.956 0.965]
 [0.96  0.956 0.946 0.96 ]
 [1.    1.    1.    1.   ]
 [0.798 0.799 0.794 0.798]
 [0.966 0.962 0.965 0.965]
 [0.951 0.956 0.961 0.956]
 [0.942 0.941 0.941 0.941]
 [0.914 0.915 0.916 0.917]
 [1.    1.    1.    1.   ]
 [0.971 0.971 0.97  0.975]
 [0.933 0.937 0.937 0.937]
 [0.932 0.928 0.932 0.928]]

Learned Delivery Robot Policy
-----------------------------
[ ↑ ] [ → ] [ ↑ ] [ S ] 
[ ↑ ] [ ↑ ] [ X ] [ ↓ ] 
[ ← ] [ → ] [ ← ] [ ↑ ] 
[ G ] [ ↑ ] [ ↑ ] [ ← ] 

Start state: 3
Path followed: [3, 7, 11, 7, 6]
Total reward: 0
Delivery unsuccessful.
